In [68]:
import logging
import os

from pymodbus.client import ModbusTcpClient
from pymodbus.framer import FramerType
from serial.tools import list_ports

from nlab_modbus.core.enums import DeviceType
from nlab_modbus.devices.geiger import GeigerDevice
from nlab_modbus.devices.psu import PSUDevice
from nlab_modbus.devices.sipm import SiPMDevice

In [69]:
def scan_devices(
    host: str,
    port: int,
    candidate_ids: range | None = None,
    scan_timeout: float = 0.02,  # 50 ms
) -> list[dict]:
    """
    Scan for Modbus devices on the given TCP host:port by reading the
    ``hardware_version`` input register (address 0, count 1) from each
    candidate device_id.

    Returns a dict mapping ``device_id -> hardware_version`` for every
    device that responded successfully.  Devices that don't answer or
    return an error are silently skipped.
    """
    if candidate_ids is None:
        candidate_ids = range(1, 17)  # 1 .. 16

    found: list[dict] = []

    for device_id in candidate_ids:
        client = ModbusTcpClient(
            host=host,
            port=port,
            framer=FramerType.RTU,
            timeout=scan_timeout,
        )
        try:
            client.connect()
            # Probe the hardware_version input register (address 0)
            result = client.read_input_registers(
                address=0,
                count=1,
                device_id=device_id,
            )
            if not result.isError():
                print("Register found:", result.registers[0])
                found.append({"type": DeviceType(int(result.registers[0])), "device_id": device_id, "host": host, "port": port})
        except Exception:
            pass  # silently skip unresponsive IDs
        finally:
            client.close()

    return found

In [70]:
HOST = "192.168.10.134"

In [71]:
all_devices = []
for port in [5001, 5002]:
    found = scan_devices(HOST, port=port)
    client = ModbusTcpClient(
        host=HOST,
        port=port,
        framer=FramerType.RTU,
        timeout=2.0,
    )
    for paramters in found:
        device_id = paramters['device_id']
        match paramters['type']:
            case DeviceType.GEIGER:
                device = GeigerDevice(client=client, device_id=device_id)
            case DeviceType.SIPM:
                device = SiPMDevice(client=client, device_id=device_id)
            case DeviceType.PSU:
                device = PSUDevice(client=client, device_id=device_id)

        device.connect()
        all_devices.append(device)


Register found: 769
Register found: 257
Register found: 513


In [72]:
def build_status_text(device) -> str:
    lines = []

    print("Device at:", device.connection_info())
    lines.append("== Status of holding registers ==")
    for i, (key, value) in enumerate(device.get_all_holding_registers().items(), start=1):
        lines.append(f"{i}. {key}: {value}")

    lines.append("")
    lines.append("== Status of input registers ==")
    for i, (key, value) in enumerate(device.get_all_input_registers().items(), start=1):
        lines.append(f"{i}. {key}: {value}")

    return "\n".join(lines) + "\n"

In [73]:
all_devices

In [74]:
for device in all_devices:
    print(device.device_type)
    # print(build_status_text(device))

AttributeError: 'PSUDevice' object has no attribute 'device_type'